In [1]:
import pandas as pd

df = pd.read_csv('../data/raw/Superstore.csv', encoding='latin-1')

print('Dimensiones:', df.shape)

Dimensiones: (51290, 28)


In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 51290 entries, 0 to 51289
Data columns (total 28 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Unnamed: 0      51290 non-null  int64  
 1   Category        51290 non-null  str    
 2   City            51290 non-null  str    
 3   Country         51290 non-null  str    
 4   Customer_ID     51290 non-null  str    
 5   Customer_Name   51290 non-null  str    
 6   Discount        51290 non-null  float64
 7   Market          51290 non-null  str    
 8   Record_Count    51290 non-null  int64  
 9   Order_Date      51290 non-null  str    
 10  Order_ID        51290 non-null  str    
 11  Order_Priority  51290 non-null  str    
 12  Product_ID      51290 non-null  str    
 13  Product_Name    51290 non-null  str    
 14  Profit          51290 non-null  float64
 15  Quantity        51290 non-null  int64  
 16  Region          51290 non-null  str    
 17  Row_ID          51290 non-null  int64  
 1

In [3]:
df.head()

,Unnamed: 0,Category,City,Country,Customer_ID,Customer_Name,Discount,Market,Record_Count,Order_Date,...,Sales,Segment,Ship_Date,Ship_Mode,Shipping_Cost,State,Sub_Category,Year,Market2,Weeknum
0,0,Office Supplies,Los Angeles,United States,LS-172304,Lycoris Saunders,0.0,US,1,2011-01-07 00:00:00.000,...,19,Consumer,2011-01-09 00:00:00.000,Second Class,4.37,California,Paper,2011,North America,2
1,1,Office Supplies,Los Angeles,United States,MV-174854,Mark Van Huff,0.0,US,1,2011-01-21 00:00:00.000,...,19,Consumer,2011-01-26 00:00:00.000,Standard Class,0.94,California,Paper,2011,North America,4
2,2,Office Supplies,Los Angeles,United States,CS-121304,Chad Sievert,0.0,US,1,2011-08-05 00:00:00.000,...,21,Consumer,2011-08-09 00:00:00.000,Standard Class,1.81,California,Paper,2011,North America,32
3,3,Office Supplies,Los Angeles,United States,CS-121304,Chad Sievert,0.0,US,1,2011-08-05 00:00:00.000,...,111,Consumer,2011-08-09 00:00:00.000,Standard Class,4.59,California,Paper,2011,North America,32
4,4,Office Supplies,Los Angeles,United States,AP-109154,Arthur Prichep,0.0,US,1,2011-09-29 00:00:00.000,...,6,Consumer,2011-10-03 00:00:00.000,Standard Class,1.32,California,Paper,2011,North America,40


In [4]:
constantes = [c for c in df.columns if df[c].nunique() ==1]
print('Columnas constantes:', constantes)

Columnas constantes: ['Record_Count']


In [5]:
df = df.drop(columns=['Unnamed: 0'] + constantes)
print('Dimensiones:', df.shape)
print('Columnas restantes', list(df.columns))

Dimensiones: (51290, 26)
Columnas restantes ['Category', 'City', 'Country', 'Customer_ID', 'Customer_Name', 'Discount', 'Market', 'Order_Date', 'Order_ID', 'Order_Priority', 'Product_ID', 'Product_Name', 'Profit', 'Quantity', 'Region', 'Row_ID', 'Sales', 'Segment', 'Ship_Date', 'Ship_Mode', 'Shipping_Cost', 'State', 'Sub_Category', 'Year', 'Market2', 'Weeknum']


In [6]:
print('Nulos por columna:')
print(df.isnull().sum()[lambda s: s > 0] if df.isnull().any().any() else 'Sin nulos')

print('\nFilas idénticas duplicadas:', df.duplicated().sum())
print('Row_ID duplicados:', df['Row_ID'].duplicated().sum())
print('Row_ID únicos:', df['Row_ID'].nunique(), 'de', len(df))

Nulos por columna:
Sin nulos

Filas idénticas duplicadas: 0
Row_ID duplicados: 0
Row_ID únicos: 51290 de 51290


In [7]:
df['Order_Date'] = pd.to_datetime(df['Order_Date'], errors='raise')
df['Ship_Date'] = pd.to_datetime(df['Ship_Date'], errors='raise')
print('Tipo Order_Date:', df['Order_Date'].dtype)
print('Tipo Ship_Date:', df['Ship_Date'].dtype)
print('Nulos Order_Date:', df['Order_Date'].isnull().sum())
print('Nulos Ship_Date:', df['Ship_Date'].isnull().sum())
print('Rango:', df['Order_Date'].min().date(), '→', df['Order_Date'].max().date())

Tipo Order_Date: datetime64[us]
Tipo Ship_Date: datetime64[us]
Nulos Order_Date: 0
Nulos Ship_Date: 0
Rango: 2011-01-01 → 2014-12-31


In [8]:
dias = (df['Ship_Date'] - df['Order_Date']).dt.days

print('Días de envío — mínimo:', dias.min())
print('Días de envío — máximo:', dias.max())
print('Días de envío — promedio:', round(dias.mean(), 2))
print('Filas con envío anterior a la orden:', (dias < 0).sum())

Días de envío — mínimo: 0
Días de envío — máximo: 7
Días de envío — promedio: 3.97
Filas con envío anterior a la orden: 0


In [9]:
assert (dias >= 0).all(), 'Hay envíos anteriores a la orden'
print('OK: ningún envío es anterior a la orden')

OK: ningún envío es anterior a la orden


In [10]:
df['Year'] = df['Order_Date'].dt.year
df['Month'] = df['Order_Date'].dt.month
df['MonthName'] = df['Order_Date'].dt.strftime('%b')
df['Quarter'] = df['Order_Date'].dt.quarter
df['YearMonth'] = df['Order_Date'].dt.to_period('M').astype(str)
df['YearMonthDate'] = df['Order_Date'].dt.to_period('M').dt.to_timestamp()
df['DaysToShip'] = (df['Ship_Date'] - df['Order_Date']).dt.days

df[['Order_Date', 'Year', 'Month', 'MonthName', 'Quarter', 'YearMonth', 'DaysToShip']].head()

,Order_Date,Year,Month,MonthName,Quarter,YearMonth,DaysToShip
0,2011-01-07,2011,1,Jan,1,2011-01,2
1,2011-01-21,2011,1,Jan,1,2011-01,5
2,2011-08-05,2011,8,Aug,3,2011-08,4
3,2011-08-05,2011,8,Aug,3,2011-08,4
4,2011-09-29,2011,9,Sep,3,2011-09,4


In [11]:
print('Años únicos:', sorted(df['Year'].unique()))

Años únicos: [np.int32(2011), np.int32(2012), np.int32(2013), np.int32(2014)]


In [12]:
ventas = df['Sales'].sum()
ganancia = df['Profit'].sum()

kpis = {
    'Ventas totales': ventas,
    'Ganancia total': ganancia,
    'Margen (%)': round(ganancia / ventas * 100, 2),
    'Órdenes únicas': df['Order_ID'].nunique(),
    'Clientes únicos (Customer_ID)': df['Customer_ID'].nunique(),
    'Cantidad vendida': int(df['Quantity'].sum()),
    'Ticket promedio por orden': round(ventas / df['Order_ID'].nunique(), 2),
    'Días promedio de envío': round(df['DaysToShip'].mean(), 2),
}

for k, v in kpis.items():
    print(f'{k}: {v:,}')

Ventas totales: 12,642,905
Ganancia total: 1,467,457.2912800002
Margen (%): 11.61
Órdenes únicas: 25,035
Clientes únicos (Customer_ID): 4,873
Cantidad vendida: 178,312
Ticket promedio por orden: 505.01
Días promedio de envío: 3.97


In [13]:
df.to_csv(
    '../data/processed/global_superstore_clean.csv',
    index=False,
    encoding='utf-8',
    sep=',',
    decimal='.'
)
print('Exportado. Filas:', len(df), '| Columnas:', df.shape[1])

Exportado. Filas: 51290 | Columnas: 32


In [14]:
print(list(df.columns))

['Category', 'City', 'Country', 'Customer_ID', 'Customer_Name', 'Discount', 'Market', 'Order_Date', 'Order_ID', 'Order_Priority', 'Product_ID', 'Product_Name', 'Profit', 'Quantity', 'Region', 'Row_ID', 'Sales', 'Segment', 'Ship_Date', 'Ship_Mode', 'Shipping_Cost', 'State', 'Sub_Category', 'Year', 'Market2', 'Weeknum', 'Month', 'MonthName', 'Quarter', 'YearMonth', 'YearMonthDate', 'DaysToShip']


In [15]:
check = pd.read_csv('../data/processed/global_superstore_clean.csv')

print('Filas:', len(check), '| original:', len(df))
print('Columnas:', check.shape[1], '| original:', df.shape[1])
print('Ventas:', round(check['Sales'].sum(), 2), '| original:', round(ventas, 2))
print('Ganancia:', round(check['Profit'].sum(), 2), '| original:', round(ganancia, 2))
print('Órdenes únicas:', check['Order_ID'].nunique(), '| original:', df['Order_ID'].nunique())
print('Clientes únicos:', check['Customer_ID'].nunique(), '| original:', df['Customer_ID'].nunique())

assert len(check) == len(df), 'Cambió el número de filas'
assert abs(check['Sales'].sum() - ventas) < 0.01, 'No coincide Sales'
assert abs(check['Profit'].sum() - ganancia) < 0.01, 'No coincide Profit'
print('Exportación verificada: todo coincide')

Filas: 51290 | original: 51290
Columnas: 32 | original: 32
Ventas: 12642905 | original: 12642905
Ganancia: 1467457.29 | original: 1467457.29
Órdenes únicas: 25035 | original: 25035
Clientes únicos: 4873 | original: 4873
Exportación verificada: todo coincide


In [16]:
anual = df.groupby('Year')[['Sales', 'Profit']].sum()
anual['Margen (%)'] = anual['Profit'] / anual['Sales'] * 100
anual['Crecimiento ventas (%)'] = anual['Sales'].pct_change() * 100
anual.round(2)

,Sales,Profit,Margen (%),Crecimiento ventas (%)
Year,,,,
2011,2259511,248940.81,11.02,NaN
2012,2677493,307415.28,11.48,18.50
2013,3405860,406935.23,11.95,27.20
2014,4300041,504165.97,11.72,26.25


In [17]:
def rentabilidad(col):
    t = df.groupby(col)[['Sales', 'Profit']].sum()
    t['Margen (%)'] = t['Profit'] / t['Sales'] * 100
    return t.sort_values('Sales', ascending=False).round(2)

rentabilidad('Category')

,Sales,Profit,Margen (%)
Category,,,
Technology,4744691,663778.73,13.99
Furniture,4110884,285204.72,6.94
Office Supplies,3787330,518473.83,13.69


In [18]:
rentabilidad('Market')

,Sales,Profit,Margen (%)
Market,,,
APAC,3585833,436000.05,12.16
EU,2938139,372829.74,12.69
US,2297354,286397.02,12.47
LATAM,2164687,221643.49,10.24
EMEA,806184,43897.97,5.45
Africa,783776,88871.63,11.34
Canada,66932,17817.39,26.62


In [19]:
sub = rentabilidad('Sub_Category').sort_values('Profit')
sub.head(5)

,Sales,Profit,Margen (%)
Sub_Category,,,
Tables,757034,-64083.39,-8.47
Fasteners,83254,11525.42,13.84
Labels,73433,15010.51,20.44
Supplies,243090,22583.26,9.29
Envelopes,170926,29601.12,17.32


In [20]:
bins = [-0.01, 0, 0.10, 0.20, 0.30, 1.0]
labels = ['0%', '1–10%', '11–20%', '21–30%', '>30%']
df['Descuento_Rango'] = pd.cut(df['Discount'], bins=bins, labels=labels)

desc = df.groupby('Descuento_Rango', observed=True).agg(
    Filas=('Profit', 'size'),
    Ventas=('Sales', 'sum'),
    Ganancia=('Profit', 'sum'),
)
desc['Margen (%)'] = desc['Ganancia'] / desc['Ventas'] * 100
desc.round(2)

,Filas,Ventas,Ganancia,Margen (%)
Descuento_Rango,,,,
0%,29009,6992734,1770695.27,25.32
1–10%,4679,1962633,338189.26,17.23
11–20%,6274,1757296,173254.84,9.86
21–30%,967,382540,-21155.61,-5.53
>30%,10361,1547702,-793526.47,-51.27


In [21]:
df[df['Discount'] > 0.30].groupby('Sub_Category')['Profit'].agg(['size', 'sum']).sort_values('sum').head(5)

,size,sum
Sub_Category,,
Tables,351,-120580.6728
Bookcases,562,-89880.3007
Phones,631,-87190.8594
Machines,341,-72125.1296
Chairs,525,-65477.7610


In [22]:
prod = df.groupby('Product_Name')['Sales'].sum().sort_values(ascending=False)
acum = prod.cumsum() / prod.sum()
n80 = int((acum < 0.80).sum() + 1)

print(f'Productos que generan el 80% de las ventas: {n80:,} de {len(prod):,} ({n80/len(prod):.1%})')

Productos que generan el 80% de las ventas: 1,066 de 3,788 (28.1%)
